<a href="https://colab.research.google.com/github/knkillname/exploraciones/blob/main/cuadernos/Jerigonza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Haciendo jerigonza en español

<a href="https://colab.research.google.com/github/knkillname/exploraciones/blob/master/cuadernos/Jerigonza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 ## Cadenas de Markov con programación funcional pura y n-gramas de caracteres

 ---

 ### 🚀 ¿De qué va esto?

 ¿Has usado el autocompletado del teclado de tu celular? ¿Has visto esos *bots*
 de Twitter que tuitean como Cervantes o como tu artista favorito? ¿Has jugado
 con ChatGPT y te has preguntado *cómo demonios hace eso*?

 Todos funcionan con una idea sorprendentemente simple:

 > **Si acabo de ver las letras «qu», ¿qué letra es más probable que venga
 > después?**

 En español, casi siempre será «e» (*que*). A veces «i» (*quien*). Casi nunca «x».
 Si en lugar de dos letras recordamos tres (« de»), probablemente siga «l» (*del*).
 Si vemos «el coronel aureliano buen», es muy probable que siga «d» (*buendía*).

 Eso es una **cadena de Markov**: un modelo estadístico que aprende, a partir de
 ejemplos, qué suele venir después de qué. No entiende gramática ni semántica —
 solo probabilidades. Y sin embargo, ¡produce jerigonza que *casi* parece español!

 ---

 ### 🎯 Lo que construirás hoy

 Al final de este cuaderno tendrás un **generador de texto estilo Markov**
 que, después de *leer* *Cien años de soledad*, producirá fragmentos como:

 > *"el coronel aureliano buendía frente al pelotón de fusilamiento había de
 > recordar aquella tarde en que su padre lo llevó a conocer el hielo"*

 — no exactamente igual, pero inquietantemente parecido.

 En el camino aprenderás:

 | Concepto | ¿Dónde? |
 |---|---|
 | $n$-gramas y ventanas deslizantes | Secciones 1–2 |
 | Contar transiciones sin bucles (`reduce` + `pairwise`) | Sección 3 |
 | Convertir conteos en probabilidades | Sección 4 |
 | Muestreo estocástico puro | Secciones 5–6 |
 | Composición de funciones sin clases ni estado | Sección 8 |
 | Visualizar el «cerebro» del modelo como un grafo | Secciones 🔬 |

 ---

 ### 📐 Un poquito de teoría (sin miedo)

 Una **cadena de Markov de orden $n$** solo recuerda las últimas $n$ letras
 para decidir la siguiente. No le importa lo que pasó antes. En símbolos:

 $$
 P(x_{t+1} \mid x_t, x_{t-1}, \ldots, x_1) = P(x_{t+1} \mid x_t, \ldots, x_{t-n+1})
 $$

 En cristiano: *«si acabo de ver "qu", solo miro "qu" para decidir — me vale
 si antes decía "Don Quijote" o "parque"»*.

 Nuestro pipeline tiene 5 pasos:

 1. **Limpiar** el texto → minúsculas, solo caracteres españoles.
 2. **Trocear** en $n$-gramas consecutivos.
 3. **Contar** cuántas veces cada $n$-grama es seguido por cada otro.
 4. **Normalizar** los conteos a probabilidades (0–1).
 5. **Muestrear** siguiendo esas probabilidades para generar texto nuevo.

 ---

 ### 🧠 Enfoque funcional

 Cada paso del pipeline es una **función pura** — sin efectos secundarios, sin
 clases, sin mutación de estado. La aleatoriedad se maneja pasando una instancia
 explícita de `random.Random`. Usamos `functools`, `itertools`, `operator` y
 `collections` de la biblioteca estándar.

 ## Dependencias e imports

 Usamos la **biblioteca estándar** de Python (≥3.10) para la lógica funcional y
 **NumPy + Matplotlib + SciPy** para visualizaciones:

 | Módulo | Uso |
 |---|---|
 | `functools` | `reduce` para composición de funciones y acumulación |
 | `itertools` | `pairwise`, `islice`, `tee`, `accumulate`, `repeat` |
 | `operator` | `itemgetter`, `truediv` para operaciones sin lambdas |
 | `collections` | `defaultdict`, `Counter` para conteo eficiente |
 | `random` | `Random` explícito como parámetro (sin estado global) |
 | `re` | Limpieza de espacios con expresión regular |
 | `numpy` | Operaciones vectorizadas para visualizaciones |
 | `matplotlib` | Gráficos: heatmaps, barras, distribuciones |
 | `networkx` | Estructura y visualización de grafos dirigidos |
 | `matplotlib.gridspec` | Layouts avanzados de gráficos |

In [ ]:
import functools
import itertools
import operator
import os
import re
import random
import urllib.request
from collections import Counter, defaultdict
from typing import Any, Callable, Dict, List, Tuple

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import networkx as nx
import numpy as np

# ── Configuración global de matplotlib ───────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 100,
        "font.size": 10,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
    }
)
import urllib.request
from collections import Counter, defaultdict
from typing import Any, Callable, Dict, List, Tuple

# ── Alias de tipos para claridad ──────────────────────────────────────────
#: Un n-grama es simplemente una cadena de n caracteres
NGram = str
#: Una transición: (siguiente n-grama, probabilidad)
Transition = Tuple[NGram, float]
#: El grafo probabilístico: cada n-grama → lista de transiciones
Graph = Dict[NGram, List[Transition]]
#: Tabla de frecuencias de transiciones (antes de normalizar)
TransitionCounts = Dict[NGram, Counter[NGram]]


 ## 1. Limpieza de texto

 La función `clean_text` es una **transformación pura** de `str → str`:

 1. Convierte a **minúsculas** con `str.lower()`.
 2. Filtra caracteres no permitidos usando `str.translate` con una tabla
    generada por `str.maketrans` — más eficiente que iterar con `filter`.
 3. Normaliza espacios múltiples a uno solo con `re.sub`.
 4. Elimina espacios sobrantes en extremos con `str.strip()`.

 > **Alternativa funcional comentada**: `filter` con `functools.partial(operator.contains, allowed)`.

In [ ]:
# ── Caracteres permitidos: alfabeto español + puntuación ──────────────────
_ALLOWED_CHARS: str = "abcdefghijklmnñopqrstuvwxyzáéíóúü.,;:!¡¿?() "


def clean_text(text: str) -> str:
    """Limpia el texto: minúsculas, solo caracteres españoles, normaliza espacios."""
    # Paso 1-2: minúsculas y reemplazo de whitespace especial
    text = text.lower().replace("\n", " ").replace("\t", " ").replace("\r", " ")

    # Paso 3: filtrar caracteres con str.translate (más rápido que iterar)
    # Construimos una tabla donde cada carácter no permitido → None (eliminar)
    _TRANSLATION_TABLE = str.maketrans(
        "", "", "".join(chr(i) for i in range(256) if chr(i) not in _ALLOWED_CHARS)
    )
    text = text.translate(_TRANSLATION_TABLE)

    # Paso 4-5: normalizar espacios y recortar extremos
    text = re.sub(r"\s+", " ", text)
    return text.strip()



 ## 2. Generación de n-gramas

 Un **$n$-grama** es una subcadena consecutiva de $n$ caracteres. Para generar
 la secuencia completa usamos el **patrón de ventana deslizante** (*sliding window*).

 En lugar de un bucle `for i in range(...)`, usamos tres herramientas de `itertools`:

 1. **`itertools.tee`**: crea $n$ iteradores independientes sobre el mismo texto.
 2. **`itertools.islice`**: desplaza cada iterador una posición más que el anterior.
 3. **`zip`**: alinea los $n$ iteradores para obtener las ventanas.

 ```
 Texto:  "hola"
 tee:     iter1 → h o l a
          iter2 → h o l a
          iter3 → h o l a
 islice:  iter1 → h o l a    (desde 0)
          iter2 →   o l a    (desde 1)
          iter3 →     l a    (desde 2)
 zip:     (h,o,l) (o,l,a)
 → n-gramas: "hol", "ola"
 ```

In [ ]:
def generate_ngrams(text: str, n: int) -> List[NGram]:
    """Genera n-gramas consecutivos con ventana deslizante (tee + islice + zip)."""
    if len(text) < n:
        return []

    # Crear n iteradores desplazados progresivamente
    iterators = itertools.tee(text, n)
    slices = (itertools.islice(it, start, None) for start, it in enumerate(iterators))

    # zip alinea los caracteres en tuplas; map(''.join) las convierte en strings
    return list(map("".join, zip(*slices)))



 ### 🔬 Visualización: ventana deslizante

 Para entender cómo funcionan los $n$-gramas, visualicemos el proceso con
 un texto de ejemplo. Cada fila es un $n$-grama extraído por la ventana.

In [ ]:
def _plot_ngram_windows(text: str, n: int = 3) -> None:
    """Dibuja las ventanas deslizantes de n-gramas sobre el texto."""
    ngrams = generate_ngrams(text, n)

    fig, (ax_top, ax_bottom) = plt.subplots(
        2,
        1,
        figsize=(12, 3),
        gridspec_kw={"height_ratios": [1, 0.4]},
    )
    fig.patch.set_facecolor("#fafafa")

    # ── Panel superior: texto con ventanas coloreadas ───────────────────
    colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(ngrams)))
    y_pos = 0.5

    for i, (ngram, color) in enumerate(zip(ngrams, colors)):
        ax_top.add_patch(
            plt.Rectangle(
                (i - 0.4, 0.1),
                0.8,
                0.8,
                facecolor=color,
                edgecolor="white",
                linewidth=1.2,
                alpha=0.85,
                zorder=2,
            )
        )
        ax_top.text(
            i,
            y_pos,
            ngram,
            ha="center",
            va="center",
            fontsize=11,
            fontfamily="monospace",
            fontweight="bold",
            color="white",
            zorder=3,
        )

    # Texto original debajo
    for j, ch in enumerate(text):
        ax_top.text(
            j,
            -0.35,
            ch,
            ha="center",
            va="center",
            fontsize=12,
            fontfamily="monospace",
            color="#333333",
        )

    ax_top.set_xlim(-1, len(text))
    ax_top.set_ylim(-0.8, 1.2)
    ax_top.set_title(
        f"Ventana deslizante de {n}-gramas sobre «{text}»",
        fontweight="bold",
        pad=12,
    )
    ax_top.axis("off")

    # ── Panel inferior: secuencia de n-gramas ───────────────────────────
    ax_bottom.barh(
        [0] * len(ngrams),
        [1] * len(ngrams),
        left=range(len(ngrams)),
        color=colors,
        height=0.7,
        edgecolor="white",
        linewidth=1,
    )
    for i, ngram in enumerate(ngrams):
        ax_bottom.text(
            i + 0.5,
            0,
            f"'{ngram}'",
            ha="center",
            va="center",
            fontsize=10,
            fontfamily="monospace",
            fontweight="bold",
            color="white",
        )

    ax_bottom.set_ylim(-0.5, 0.5)
    ax_bottom.set_xlim(-0.5, len(ngrams) + 0.5)
    ax_bottom.set_title("Secuencia resultante", fontweight="bold", pad=8)
    ax_bottom.axis("off")

    plt.tight_layout()
    plt.show()


_plot_ngram_windows("aureliano buendía", n=3)


 ## 3. Construcción del modelo de transiciones

 Dada la secuencia de $n$-gramas, necesitamos contar cuántas veces cada $n$-grama
 es seguido por cada posible siguiente $n$-grama.

 ### Enfoque funcional

 Usamos **`itertools.pairwise`** (Python ≥3.10) para iterar pares adyacentes
 `(ngram_i, ngram_{i+1})` y **`functools.reduce`** para acumular los conteos en
 un `defaultdict(Counter)` — sin bucles `for` explícitos ni mutación de variables.

 ```
 n-gramas:  ['hol', 'ola', 'la ', 'a m', ' mu', 'mun', 'und', 'ndo']
 pairwise:  ('hol','ola') → ('ola','la ') → ('la ','a m') → ...
 reduce:    defaultdict(Counter)  ←  acumula cada par
 ```

 La función `_add_transition` usa `Counter.__setitem__` para incrementar:
 `acc[actual][siguiente] += 1` — sin `or acc` porque ya retornamos `acc`.

In [ ]:
def _add_transition(
    acc: TransitionCounts, pair: tuple[NGram, NGram]
) -> TransitionCounts:
    """Incrementa el contador de transición `pair[0] → pair[1]`."""
    acc[pair[0]][pair[1]] += 1
    return acc


def build_transitions(ngrams: List[NGram]) -> TransitionCounts:
    """Cuenta transiciones entre n-gramas adyacentes con pairwise + reduce."""
    if len(ngrams) < 2:
        raise ValueError(
            f"Se necesitan al menos 2 n-gramas para construir transiciones "
            f"(recibidos: {len(ngrams)})"
        )

    return functools.reduce(
        _add_transition,
        itertools.pairwise(ngrams),
        defaultdict(Counter),
    )



 ### 🔬 Visualización: matriz de transiciones

 Construimos un modelo pequeño con una frase de ejemplo y visualizamos
 la matriz de transiciones como un *heatmap*. El color indica la frecuencia
 con que cada $n$-grama (fila) transita a otro (columna).

In [ ]:
def _plot_transition_heatmap(text: str, n: int = 2) -> None:
    """Heatmap de la matriz de transiciones para un texto pequeño."""
    ngrams = generate_ngrams(clean_text(text), n)
    transitions = build_transitions(ngrams)
    nodes = sorted(transitions.keys())

    if len(nodes) < 2:
        print("Se necesitan al menos 2 n-gramas únicos.")
        return

    # Construir matriz de co-ocurrencia
    idx = {node: i for i, node in enumerate(nodes)}
    size = len(nodes)
    matrix = np.zeros((size, size))

    for src, counter in transitions.items():
        for dst, count in counter.items():
            if dst in idx:
                matrix[idx[src], idx[dst]] = count

    fig, ax = plt.subplots(figsize=(max(6, size * 0.7), max(5, size * 0.6)))
    im = ax.imshow(matrix, cmap="YlOrRd", aspect="auto", vmin=0)

    ax.set_xticks(range(size))
    ax.set_yticks(range(size))
    ax.set_xticklabels(nodes, fontfamily="monospace", fontsize=9, rotation=45)
    ax.set_yticklabels(nodes, fontfamily="monospace", fontsize=9)
    ax.set_title(
        f"Matriz de transiciones — {n}-gramas\n«{text}»",
        fontweight="bold",
        pad=12,
    )
    ax.set_xlabel("n-grama siguiente →", labelpad=8)
    ax.set_ylabel("← n-grama actual", labelpad=8)

    # Anotar valores
    for i in range(size):
        for j in range(size):
            if matrix[i, j] > 0:
                color = "white" if matrix[i, j] > matrix.max() / 2 else "#333333"
                ax.text(
                    j,
                    i,
                    int(matrix[i, j]),
                    ha="center",
                    va="center",
                    fontsize=8,
                    fontweight="bold",
                    color=color,
                )

    cbar = plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
    cbar.set_label("Frecuencia", fontsize=9)
    plt.tight_layout()
    plt.show()


_plot_transition_heatmap("muchos años después, frente al pelotón de fusilamiento", n=2)


 ## 4. Normalización de probabilidades

 Cada nodo del grafo tiene $k$ aristas salientes con frecuencias $c_1, c_2, \ldots, c_k$.
 La probabilidad de cada transición es:

 $$P(\text{siguiente}_j \mid \text{actual}) = \frac{c_j}{\sum_{i=1}^{k} c_i}$$

 Implementado con **dict comprehension** + **walrus operator** (`:=`) para calcular
 la suma una sola vez por nodo, y `operator.truediv` como alternativa didáctica
 (aunque en este caso usamos `/` directamente por legibilidad).

In [ ]:
def normalize_probabilities(transitions: TransitionCounts) -> Graph:
    """Convierte frecuencias de transición en probabilidades (0-1) por nodo."""
    return {
        current: [
            (next_ngram, count / total) for next_ngram, count in next_counter.items()
        ]
        for current, next_counter in transitions.items()
        if (total := sum(next_counter.values())) > 0
    }


def to_networkx(graph: Graph) -> nx.DiGraph:
    """Convierte el grafo de Markov a un DiGraph de NetworkX."""
    g = nx.DiGraph()
    for src, transitions in graph.items():
        for dst, prob in transitions:
            g.add_edge(src, dst, weight=prob)
    return g



 ### 🔬 Visualización: distribuciones de probabilidad

 Para los nodos con más aristas del modelo de ejemplo, mostramos las
 probabilidades de transición como barras horizontales.

In [ ]:
def _plot_top_probabilities(graph: Graph, top_k: int = 4) -> None:
    """Gráfico de barras horizontales con las probabilidades de los top nodos."""
    if not graph:
        print("Grafo vacío.")
        return

    # Seleccionar nodos con más aristas
    top_nodes = sorted(graph.items(), key=lambda kv: len(kv[1]), reverse=True)[:top_k]
    n_cols = min(2, len(top_nodes))
    n_rows = (len(top_nodes) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 3 * n_rows))
    if n_rows * n_cols == 1:
        axes = np.array([[axes]])
    axes = np.atleast_2d(axes)

    colors = plt.cm.Set2(np.linspace(0, 1, 8))

    for idx, (node, transitions) in enumerate(top_nodes):
        ax = axes[idx // n_cols, idx % n_cols]
        trans_sorted = sorted(transitions, key=lambda t: t[1], reverse=True)[:8]

        labels = [t[0] for t in trans_sorted]
        probs = [t[1] for t in trans_sorted]

        bars = ax.barh(
            range(len(labels)), probs, color=colors[: len(labels)], height=0.6
        )
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontfamily="monospace", fontsize=10)
        ax.invert_yaxis()
        ax.set_xlim(0, max(probs) * 1.25 if probs else 1)
        ax.set_title(f"Nodo: '{node}'", fontweight="bold", fontfamily="monospace")
        ax.set_xlabel("Probabilidad")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        for bar, prob in zip(bars, probs):
            ax.text(
                bar.get_width() + 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{prob:.2f}",
                va="center",
                fontsize=9,
                fontweight="bold",
            )

    # Ocultar axes sobrantes
    for idx in range(len(top_nodes), n_rows * n_cols):
        axes[idx // n_cols, idx % n_cols].set_visible(False)

    fig.suptitle("Distribuciones de probabilidad por nodo", fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()


demo_text = (
    "muchos años después, frente al pelotón de fusilamiento, "
    "el coronel aureliano buendía había de recordar aquella tarde "
    "remota en que su padre lo llevó a conocer el hielo. macondo "
    "era entonces una aldea de veinte casas de barro y cañabrava "
    "construidas a la orilla de un río de aguas diáfanas que se "
    "precipitaban por un lecho de piedras pulidas, blancas y enormes "
    "como huevos prehistóricos."
)
demo_ngrams = generate_ngrams(clean_text(demo_text), n=2)
demo_transitions = build_transitions(demo_ngrams)
demo_model = normalize_probabilities(demo_transitions)
_plot_top_probabilities(demo_model)


 ## 5. Muestreo estocástico puro

 Para generar texto, necesitamos elegir el siguiente $n$-grama según la distribución
 de probabilidad aprendida. La función `sample_next`:

 1. Recibe el **grafo**, el **$n$-grama actual** y una instancia de **`random.Random`**.
 2. Si el $n$-grama no está en el grafo (nunca visto), elige uno aleatorio como *fallback*.
 3. Usa `random.Random.choices` con pesos para muestrear según la distribución.

 La pureza se mantiene porque `random.Random` es un **generador explícito** —
 la misma semilla produce siempre la misma secuencia, haciendo la función
 determinista para una `rng` dada.

In [ ]:
def sample_next(
    graph: Graph,
    current_ngram: NGram,
    rng: random.Random,
) -> NGram:
    """Muestrea el siguiente n-grama según la distribución de probabilidad del grafo."""
    if current_ngram not in graph:
        # Fallback: elegir un nodo aleatorio del grafo, o devolver el mismo
        return rng.choice(list(graph.keys())) if graph else current_ngram

    # Separar claves y pesos con zip(*) — patrón funcional clásico
    transitions = graph[current_ngram]
    next_ngrams, probabilities = zip(*transitions)

    return rng.choices(next_ngrams, weights=probabilities, k=1)[0]



 ### 🔬 Visualización: grafo dirigido con NetworkX

 Convertimos el modelo a un **`nx.DiGraph`** con `to_networkx()` y lo
 visualizamos con el layout `kamada_kawai`. El grosor de cada arista es
 proporcional a su probabilidad, y el tamaño del nodo refleja su grado
 de salida. Para no saturar, mostramos solo las aristas más probables.

In [ ]:
def _plot_markov_graph(
    graph: Graph,
    title: str = "Grafo de Markov",
    max_edges_display: int = 60,
) -> None:
    """Visualiza el grafo de Markov como grafo dirigido con NetworkX."""
    if not graph:
        print("Grafo vacío.")
        return

    g = to_networkx(graph)

    # ── Seleccionar aristas más probables para no saturar ───────────────
    edges_sorted = sorted(
        g.edges(data=True), key=lambda e: e[2]["weight"], reverse=True
    )
    top_edges = edges_sorted[:max_edges_display]

    viz = nx.DiGraph()
    for u, v, d in top_edges:
        viz.add_edge(u, v, weight=d["weight"])

    # ── Layout ──────────────────────────────────────────────────────────
    pos = nx.kamada_kawai_layout(viz)

    fig, ax = plt.subplots(figsize=(12, 10))

    # Tamaño y color de nodo proporcional al grado de salida
    degrees = [viz.out_degree(n) for n in viz.nodes()]
    node_sizes = [300 + 80 * d for d in degrees]

    nx.draw_networkx_nodes(
        viz,
        pos,
        ax=ax,
        node_size=node_sizes,
        node_color=degrees,
        cmap=plt.cm.YlOrRd,
        edgecolors="#333333",
        linewidths=0.8,
        alpha=0.9,
    )

    nx.draw_networkx_labels(
        viz,
        pos,
        ax=ax,
        font_size=8,
        font_family="monospace",
        font_weight="bold",
    )

    # Aristas con grosor y opacidad proporcionales a probabilidad
    edge_weights = [d["weight"] for _, _, d in viz.edges(data=True)]
    if edge_weights:
        max_w = max(edge_weights)
        edge_widths = [0.8 + 4.5 * (w / max_w) for w in edge_weights]
        edge_alphas = [0.25 + 0.7 * (w / max_w) for w in edge_weights]
    else:
        edge_widths, edge_alphas = [], []

    nx.draw_networkx_edges(
        viz,
        pos,
        ax=ax,
        width=edge_widths,
        alpha=edge_alphas,
        edge_color="#555555",
        arrows=True,
        arrowsize=12,
        arrowstyle="-|>",
        connectionstyle="arc3,rad=0.15",
    )

    # ── Leyenda de grosor de arista ─────────────────────────────────────
    if edge_weights:
        legend_probs = [max_w, max_w * 0.5, max_w * 0.1]
        for p, label in zip(legend_probs, ["alta", "media", "baja"]):
            lw = 0.8 + 4.5 * (p / max_w)
            ax.plot(
                [], [], color="#555555", linewidth=lw, label=f"p ≈ {p:.2f} ({label})"
            )
        ax.legend(
            title="Probabilidad de transición",
            fontsize=7,
            title_fontsize=8,
            loc="lower left",
        )

    ax.set_title(
        f"{title}\n"
        f"{viz.number_of_nodes()} nodos, {viz.number_of_edges()} aristas "
        f"({g.number_of_edges()} totales, mostrando top {max_edges_display})",
        fontweight="bold",
        pad=12,
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()


_plot_markov_graph(demo_model, "Cien años de soledad — grafo de Markov (n=2)")


 ## 6. Generación de texto

 El proceso generativo usa **`itertools.accumulate`** con **`itertools.repeat`**
 para producir la secuencia de $n$-gramas de forma funcional:

 1. **Semilla**: se elige un $n$-grama inicial (dado o aleatorio).
 2. **Acumulación**: `accumulate` aplica `sample_next` repetidamente,
    cada nuevo estado se convierte en la entrada del siguiente paso.
 3. **Proyección**: extraemos el último carácter de cada $n$-grama con
    `operator.itemgetter(-1)` y concatenamos.

 ```
 accumulate:  seed → ngram₁ → ngram₂ → ngram₃ → ...
 itemgetter:          [-1]     [-1]     [-1]
 resultado:   "hol" + "a"   + " "   + "m"   + ...
 ```

In [ ]:
def generate_text(
    graph: Graph,
    n: int,
    seed: str | None = None,
    max_length: int = 100,
    rng: random.Random | None = None,
) -> str:
    """Genera texto muestreando el modelo de Markov con accumulate + repeat."""
    if not graph:
        return ""

    rng = rng if rng is not None else random.Random()

    # ── Semilla inicial ──────────────────────────────────────────────────
    if seed is None:
        current = rng.choice(list(graph.keys()))
    else:
        current = seed[:n] if len(seed) >= n else seed.ljust(n)[:n]

    # ── Generar secuencia de n-gramas con accumulate ──────────────────────
    # accumulate aplica sample_next repetidamente: cada resultado es la
    # entrada del siguiente paso. repeat(None) proporciona el "combustible".
    def _step(current: str, _: Any) -> str:
        return sample_next(graph, current, rng)

    ngram_sequence = itertools.accumulate(
        itertools.repeat(None, max_length - n),
        _step,
        initial=current,
    )

    # ── Extraer caracteres y unir ─────────────────────────────────────────
    # El primer n-grama se inserta completo; del resto solo el último carácter
    first_ngram = current
    tail_chars = map(
        operator.itemgetter(-1),
        itertools.islice(ngram_sequence, 1, None),
    )

    return first_ngram + "".join(tail_chars)



 ## 7. Estadísticas del modelo

 Separamos la **computación** de la **presentación**: `compute_statistics` retorna
 un diccionario inmutable con los datos, y `format_statistics` lo convierte en
 texto legible. Esto mantiene la pureza y permite reutilizar los datos para
 visualizaciones, pruebas unitarias, etc.

In [ ]:
def compute_statistics(graph: Graph, n: int) -> dict[str, Any]:
    """Calcula estadísticas descriptivas del grafo: nodos, aristas, promedio, top 5."""
    if not graph:
        return {
            "n": n,
            "total_nodes": 0,
            "total_edges": 0,
            "avg_edges_per_node": 0.0,
            "top_nodes": [],
        }

    total_nodes = len(graph)
    # Suma de aristas usando map(len) — sin bucles explícitos
    total_edges = sum(map(len, graph.values()))

    # Top 5 nodos por número de aristas salientes
    def _edge_count(item: tuple[NGram, List[Transition]]) -> int:
        return len(item[1])

    top_nodes = sorted(
        graph.items(),
        key=_edge_count,
        reverse=True,
    )[:5]

    top_nodes_data = [
        {
            "node": node,
            "edge_count": len(transitions),
            "top_transitions": sorted(
                transitions,
                key=operator.itemgetter(1),  # ordenar por probabilidad
                reverse=True,
            )[:3],
        }
        for node, transitions in top_nodes
    ]

    return {
        "n": n,
        "total_nodes": total_nodes,
        "total_edges": total_edges,
        "avg_edges_per_node": total_edges / total_nodes if total_nodes > 0 else 0.0,
        "top_nodes": top_nodes_data,
    }


def format_statistics(stats: dict[str, Any]) -> str:
    """Formatea las estadísticas del modelo como texto legible."""
    if stats["total_nodes"] == 0:
        return "⚠️  El modelo está vacío (sin datos de entrenamiento)."

    lines = [
        "=" * 50,
        "📊  Estadísticas del Modelo de Markov",
        "=" * 50,
        f"  Tamaño de n-gramas (n):      {stats['n']}",
        f"  Nodos únicos:                {stats['total_nodes']}",
        f"  Aristas totales:             {stats['total_edges']}",
        f"  Promedio aristas por nodo:   {stats['avg_edges_per_node']:.2f}",
        "",
        "─" * 50,
        "  🔝  Top 5 nodos con más transiciones",
        "─" * 50,
    ]

    for i, node_data in enumerate(stats["top_nodes"], 1):
        lines.append(
            f"\n  {i}. Nodo: '{node_data['node']}' ({node_data['edge_count']} aristas)"
        )
        for next_ngram, prob in node_data["top_transitions"]:
            lines.append(f"       → '{next_ngram}'  (p = {prob:.4f})")

    lines.append("\n" + "=" * 50)
    return "\n".join(lines)



 ### 🔬 Visualización: dashboard de estadísticas

 Convertimos las estadísticas en gráficos: distribución de aristas por nodo
 y top nodos con sus probabilidades principales.

In [ ]:
def _plot_stats_dashboard(stats: dict[str, Any], graph: Graph) -> None:
    """Dashboard visual con histograma de aristas y top nodos."""
    fig = plt.figure(figsize=(14, 6))
    gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1, 1.3])
    fig.patch.set_facecolor("#fafafa")

    # ── Panel izquierdo: histograma de aristas por nodo ──────────────────
    ax1 = fig.add_subplot(gs[0])
    edge_counts = sorted(map(len, graph.values()), reverse=True)

    if edge_counts:
        bins = np.logspace(0, np.log10(max(edge_counts) + 1), 30)
        ax1.hist(edge_counts, bins=bins, color="#2a9d8f", edgecolor="white", alpha=0.85)
        ax1.set_xscale("log")
        ax1.set_xlabel("Aristas salientes (escala log)")
        ax1.set_ylabel("Número de nodos")
        ax1.set_title("Distribución de aristas por nodo", fontweight="bold")
        ax1.spines["top"].set_visible(False)
        ax1.spines["right"].set_visible(False)

        # Líneas de referencia
        ax1.axvline(
            np.median(edge_counts),
            color="#e76f51",
            linestyle="--",
            linewidth=1.5,
            label=f"Mediana = {np.median(edge_counts):.0f}",
        )
        ax1.axvline(
            np.mean(edge_counts),
            color="#264653",
            linestyle="-",
            linewidth=1.5,
            label=f"Media = {np.mean(edge_counts):.1f}",
        )
        ax1.legend(fontsize=9)

    # ── Panel derecho: top nodos con barras apiladas ─────────────────────
    ax2 = fig.add_subplot(gs[1])
    top_nodes = stats["top_nodes"][:6]

    if top_nodes:
        colors_top = plt.cm.viridis(np.linspace(0.1, 0.9, 3))
        y_labels = []
        for i, nd in enumerate(top_nodes):
            y_labels.append(nd["node"])
            left = 0
            for j, (ng, prob) in enumerate(nd["top_transitions"]):
                ax2.barh(
                    i,
                    prob,
                    left=left,
                    color=colors_top[j % 3],
                    height=0.55,
                    edgecolor="white",
                    linewidth=0.8,
                    label=ng if i == 0 else "",
                )
                left += prob

        ax2.set_yticks(range(len(y_labels)))
        ax2.set_yticklabels(y_labels, fontfamily="monospace", fontsize=10)
        ax2.invert_yaxis()
        ax2.set_xlim(0, 1)
        ax2.set_xlabel("Probabilidad acumulada")
        ax2.set_title("Top nodos — transiciones principales", fontweight="bold")
        ax2.spines["top"].set_visible(False)
        ax2.spines["right"].set_visible(False)

        # Leyenda sin duplicados
        handles, labels_ = ax2.get_legend_handles_labels()
        by_label = dict(zip(labels_, handles))
        ax2.legend(by_label.values(), by_label.keys(), fontsize=8, loc="lower right")

    fig.suptitle(
        f"Dashboard del modelo — {stats['total_nodes']:,} nodos, "
        f"{stats['total_edges']:,} aristas, "
        f"promedio {stats['avg_edges_per_node']:.2f} aristas/nodo",
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()


_plot_stats_dashboard(
    compute_statistics(demo_model, n=2),
    demo_model,
)


 ## 8. Composición funcional del pipeline

 El corazón del estilo funcional: **componer funciones** en lugar de secuenciar
 instrucciones. `pipe` encadena funciones de izquierda a derecha usando
 `functools.reduce`:

 ```
 pipe(x, f, g, h)  ≡  h(g(f(x)))
 ```

 Y `create_model` es la composición declarativa del pipeline completo:

 ```
 create_model = pipe ∘ (clean_text → generate_ngrams → build_transitions → normalize_probabilities)
 ```

In [ ]:
def _apply(value: Any, fn: Callable[..., Any]) -> Any:
    """Aplica una función a un valor: fn(value)."""
    return fn(value)


def pipe(value: Any, *functions: Callable[..., Any]) -> Any:
    """Compone funciones secuencialmente: pipe(x, f, g, h) ≡ h(g(f(x)))."""
    return functools.reduce(_apply, functions, value)


def create_model(text: str, n: int = 3) -> Graph:
    """Pipeline declarativo: clean_text → generate_ngrams → build_transitions → normalize."""
    # Currificamos generate_ngrams con n fijo usando functools.partial
    generate_with_n = functools.partial(generate_ngrams, n=n)

    return pipe(
        text,
        clean_text,
        generate_with_n,
        build_transitions,
        normalize_probabilities,
    )



 ## 9. Descarga inteligente con caché

 Para la demostración usaremos *"Don Quijote de la Mancha"* (Project Gutenberg,
 texto `pg2000.txt`). La función `download_pg2000` implementa un patrón de
 **caché funcional**:

 1. Si el archivo ya existe en disco → leer y retornar (sin red).
 2. Si no existe → descargar de la URL raw de GitHub, guardar en disco, retornar.

 Esto evita descargas repetidas y mantiene el efecto de red aislado en una sola función.

In [ ]:
# ── URL raw del texto en GitHub ──────────────────────────────────────────
_PG2000_URL: str = (
    "https://raw.githubusercontent.com/knkillname/exploraciones/"
    "master/datos/pg2000.txt"
)
_CACHE_DIR: str = "datos"
_CACHE_FILE: str = os.path.join(_CACHE_DIR, "pg2000.txt")


def download_pg2000(
    url: str = _PG2000_URL,
    cache_path: str = _CACHE_FILE,
) -> str:
    """Descarga el texto de Don Quijote con caché en disco para evitar re-descargas."""
    # ── Verificar caché ──────────────────────────────────────────────────
    if os.path.exists(cache_path):
        with open(cache_path, encoding="utf-8") as f:
            return f.read()

    # ── Descargar ─────────────────────────────────────────────────────────
    print(f"📥  Descargando {url} ...")
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)

    with urllib.request.urlopen(url) as response:
        content = response.read().decode("utf-8")

    # ── Guardar en caché ──────────────────────────────────────────────────
    with open(cache_path, "w", encoding="utf-8") as f:
        f.write(content)

    print(f"💾  Archivo guardado en: {cache_path}")
    return content



 ## 10. Demostración

 Pipeline de extremo a extremo ejecutado paso a paso, con resultados visibles
 en cada etapa. Ajusta los parámetros y re-ejecuta las celdas a tu gusto.

 ### 10.1 Carga de datos

 Descargamos *Don Quijote* (con caché) y recortamos a los primeros 200 000
 caracteres para un entrenamiento rápido.

In [ ]:
def load_training_text(max_chars: int | None = None) -> str:
    """Descarga (con caché) y recorta el texto de entrenamiento."""
    return download_pg2000()[slice(None, max_chars)]


text = load_training_text()
print(f"📚  {len(text):,} caracteres cargados.")


 ### 10.2 Entrenamiento del modelo

 Construimos el grafo de Markov con $n = 3$. El pipeline funcional compone
 `clean_text → generate_ngrams → build_transitions → normalize_probabilities`.

In [ ]:
N = 3
model = create_model(text, n=N)

stats = compute_statistics(model, n=N)
print(format_statistics(stats))


 ### 10.3 Textos generados

 Generamos muestras con semilla fija (`random.Random(42)`) para reproducibilidad.
 Cada llamada a `generate_text` recibe el grafo y el generador como parámetros
 explícitos — sin estado global.

In [ ]:
rng = random.Random()

samples = [
    ("el ingenioso", "«El ingenioso...»"),
    ("en un lugar", "«En un lugar...»"),
    (None, "(semilla aleatoria)"),
]

for seed, label in samples:
    generated = generate_text(
        graph=model,
        n=N,
        seed=seed,
        max_length=300,
        rng=rng,
    )
    print(f"🎲  Semilla: {label}")
    print(f"    {generated}")
    print()


 ### 🔬 Visualización: distribuciones de caracteres

 Comparamos la frecuencia de caracteres en el texto original (*Don Quijote*)
 contra el texto generado por el modelo. Un buen modelo debería preservar
 aproximadamente la distribución.

In [ ]:
def _plot_char_distributions(original: str, generated: str) -> None:
    """Compara histogramas de frecuencia de caracteres: original vs generado."""
    from collections import Counter as _Counter

    orig_counts = _Counter(original)
    gen_counts = _Counter(generated)

    # Usar solo caracteres presentes en ambos
    all_chars = sorted(set(orig_counts) | set(gen_counts))
    # Filtrar espacio y puntuación común para claridad
    chars = [c for c in all_chars if c not in {" "}][:30]

    orig_freq = np.array([orig_counts.get(c, 0) for c in chars], dtype=float)
    gen_freq = np.array([gen_counts.get(c, 0) for c in chars], dtype=float)
    orig_freq /= orig_freq.sum()
    gen_freq /= gen_freq.sum()

    x = np.arange(len(chars))
    width = 0.35

    fig, ax = plt.subplots(figsize=(14, 5))
    bars1 = ax.bar(
        x - width / 2,
        orig_freq,
        width,
        label="Original (Don Quijote)",
        color="#2a9d8f",
        edgecolor="white",
        alpha=0.85,
    )
    bars2 = ax.bar(
        x + width / 2,
        gen_freq,
        width,
        label="Generado por Markov",
        color="#e76f51",
        edgecolor="white",
        alpha=0.85,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(chars, fontfamily="monospace", fontsize=10)
    ax.set_ylabel("Frecuencia relativa")
    ax.set_title("Distribución de caracteres: Original vs Generado", fontweight="bold")
    ax.legend(fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Anotar diferencia para los más dispares
    diffs = np.abs(orig_freq - gen_freq)
    for i in np.argsort(diffs)[-3:]:
        ax.annotate(
            f"Δ={diffs[i]:.3f}",
            (x[i], max(orig_freq[i], gen_freq[i]) + 0.005),
            ha="center",
            fontsize=8,
            color="#333333",
            fontweight="bold",
        )

    plt.tight_layout()
    plt.show()


# Generar suficiente texto para comparar
rng_viz = random.Random(123)
gen_text = generate_text(model, n=N, seed="en un lugar", max_length=3000, rng=rng_viz)
_plot_char_distributions(text, gen_text)


 ## 📚 Referencias y lecturas adicionales

 - **Cadenas de Markov**: [Markov chain — Wikipedia](https://en.wikipedia.org/wiki/Markov_chain)
 - **Programación funcional en Python**:
   - [`functools` — Higher-order functions](https://docs.python.org/3/library/functools.html)
   - [`itertools` — Iterator building blocks](https://docs.python.org/3/library/itertools.html)
   - [`operator` — Standard operators as functions](https://docs.python.org/3/library/operator.html)
 - **Project Gutenberg**: [Don Quijote (pg2000)](https://www.gutenberg.org/ebooks/2000)
 - **Programación literaria**: [Literate programming — Wikipedia](https://en.wikipedia.org/wiki/Literate_programming)
 - **VS Code Python Interactive**: [Python Interactive Window](https://code.visualstudio.com/docs/python/jupyter-support-py)

 ---
 > *"La pureza funcional no es un fin en sí mismo, sino un medio para razonar
 > sobre el código con confianza."*
 > — Anónimo funcional